In [25]:
import gc
import numpy as np
import pandas as pd
import joblib as joblib
import json 
from time import perf_counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split

RUTA_PROYECTO = Path("..")

RUTA_VECTORIZACION = (
    RUTA_PROYECTO
    / "models"
    / "primera_iteracion"
    / "vectorizacion"
)

datos_train = pd.read_parquet(
    RUTA_VECTORIZACION / "datos_train.parquet"
)

X_train = datos_train[
    "review_text_processed"
]

y_train = datos_train[
    "sentiment"
]

print("Textos:", f"{len(X_train):,}")
print("Etiquetas:", f"{len(y_train):,}")
print("Tipo del primer texto:", type(X_train.iloc[0]))


Textos: 1,188,274
Etiquetas: 1,188,274
Tipo del primer texto: <class 'str'>


In [22]:
(
    X_train_interno,
    X_validacion,
    y_train_interno,
    y_validacion
) = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

In [4]:
print("DIVISIÓN INTERNA")
print("----------------")

print(
    "Entrenamiento interno:",
    f"{len(X_train_interno):,}"
)

print(
    "Validación:",
    f"{len(X_validacion):,}"
)

print(
    "Etiquetas de entrenamiento interno:",
    f"{len(y_train_interno):,}"
)

print(
    "Etiquetas de validación:",
    f"{len(y_validacion):,}"
)

print(
    "¿Coincide entrenamiento interno?",
    len(X_train_interno)
    == len(y_train_interno)
)

print(
    "¿Coincide validación?",
    len(X_validacion)
    == len(y_validacion)
)

DIVISIÓN INTERNA
----------------
Entrenamiento interno: 950,619
Validación: 237,655
Etiquetas de entrenamiento interno: 950,619
Etiquetas de validación: 237,655
¿Coincide entrenamiento interno? True
¿Coincide validación? True


In [5]:
vectorizador_interno = TfidfVectorizer(
    max_features=50_000,
    min_df=5,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True,
    dtype=np.float32
)

print("Vectorizando entrenamiento interno...")

inicio_vectorizacion = perf_counter()

X_train_interno_tfidf = (
    vectorizador_interno.fit_transform(
        X_train_interno
    )
)

print("Transformando validación...")

X_validacion_tfidf = (
    vectorizador_interno.transform(
        X_validacion
    )
)

tiempo_vectorizacion_interna = (
    perf_counter() - inicio_vectorizacion
)

print("\nVECTORIZACIÓN COMPLETADA")
print("------------------------")
print(
    "Entrenamiento interno:",
    X_train_interno_tfidf.shape
)
print(
    "Validación:",
    X_validacion_tfidf.shape
)
print(
    "Tiempo:",
    f"{tiempo_vectorizacion_interna / 60:.2f} minutos"
)

Vectorizando entrenamiento interno...
Transformando validación...

VECTORIZACIÓN COMPLETADA
------------------------
Entrenamiento interno: (950619, 50000)
Validación: (237655, 50000)
Tiempo: 2.18 minutos


In [6]:
configuraciones_rl = [
    {
        "nombre": "regularizacion_fuerte",
        "C": 0.1,
        "class_weight": None,
        "tol": 1e-3,
        "max_iter": 500
    },
    {
        "nombre": "regularizacion_intermedia",
        "C": 0.5,
        "class_weight": None,
        "tol": 1e-3,
        "max_iter": 500
    },
    {
        "nombre": "clases_balanceadas",
        "C": 0.5,
        "class_weight": "balanced",
        "tol": 1e-3,
        "max_iter": 500
    }
]

In [9]:
resultados_ajuste_rl = []
informes_ajuste_rl = {}

for numero, configuracion in enumerate(
    configuraciones_rl,
    start=1
):
    print(
        f"\nCONFIGURACIÓN {numero}/"
        f"{len(configuraciones_rl)}"
    )
    print(configuracion["nombre"])
    print("-" * 50)

    # Crear el modelo
    modelo_temporal = LogisticRegression(
        C=configuracion["C"],
        class_weight=configuracion["class_weight"],
        solver="saga",
        tol=configuracion["tol"],
        max_iter=configuracion["max_iter"],
        random_state=42,
        verbose=0
    )

    # Entrenar
    print("Entrenando modelo...")

    inicio_entrenamiento = perf_counter()

    modelo_temporal.fit(
        X_train_interno_tfidf,
        y_train_interno
    )

    tiempo_entrenamiento = (
        perf_counter() - inicio_entrenamiento
    )

    # Predecir
    print("Generando predicciones...")

    inicio_prediccion = perf_counter()

    predicciones_validacion = (
        modelo_temporal.predict(
            X_validacion_tfidf
        )
    )

    tiempo_prediccion = (
        perf_counter() - inicio_prediccion
    )

    # Calcular métricas generales
    accuracy = accuracy_score(
        y_validacion,
        predicciones_validacion
    )

    precision_macro = precision_score(
        y_validacion,
        predicciones_validacion,
        average="macro",
        zero_division=0
    )

    recall_macro = recall_score(
        y_validacion,
        predicciones_validacion,
        average="macro",
        zero_division=0
    )

    f1_macro = f1_score(
        y_validacion,
        predicciones_validacion,
        average="macro",
        zero_division=0
    )

    # Obtener métricas por clase
    informe = classification_report(
        y_validacion,
        predicciones_validacion,
        labels=[
            "negativo",
            "neutral",
            "positivo"
        ],
        output_dict=True,
        zero_division=0
    )

    informes_ajuste_rl[
        configuracion["nombre"]
    ] = informe

    # Comprobar convergencia
    iteraciones = int(
        modelo_temporal.n_iter_.max()
    )

    convergencia = (
        iteraciones
        < configuracion["max_iter"]
    )

    # Guardar resultados
    resultados_ajuste_rl.append({
        "Configuración": configuracion["nombre"],
        "C": configuracion["C"],
        "class_weight": str(
            configuracion["class_weight"]
        ),
        "tol": configuracion["tol"],
        "max_iter": configuracion["max_iter"],
        "Iteraciones": iteraciones,
        "Convergencia": convergencia,
        "Accuracy": accuracy,
        "Precision macro": precision_macro,
        "Recall macro": recall_macro,
        "F1 macro": f1_macro,
        "F1 negativo": informe[
            "negativo"
        ]["f1-score"],
        "F1 neutral": informe[
            "neutral"
        ]["f1-score"],
        "F1 positivo": informe[
            "positivo"
        ]["f1-score"],
        "Tiempo entrenamiento": tiempo_entrenamiento,
        "Tiempo predicción": tiempo_prediccion
    })

    # Mostrar resultados
    print("\nRESULTADOS")
    print("----------")
    print(f"Accuracy:        {accuracy:.4f}")
    print(f"Precision macro: {precision_macro:.4f}")
    print(f"Recall macro:    {recall_macro:.4f}")
    print(f"F1 macro:        {f1_macro:.4f}")

    print(
        "F1 negativo:",
        f"{informe['negativo']['f1-score']:.4f}"
    )

    print(
        "F1 neutral:",
        f"{informe['neutral']['f1-score']:.4f}"
    )

    print(
        "F1 positivo:",
        f"{informe['positivo']['f1-score']:.4f}"
    )

    print(
        "Iteraciones realizadas:",
        iteraciones
    )

    print(
        "¿Convergencia alcanzada?",
        convergencia
    )

    print(
        "Tiempo de entrenamiento:",
        f"{tiempo_entrenamiento / 60:.2f} minutos"
    )

    print(
        "Tiempo de predicción:",
        f"{tiempo_prediccion:.2f} segundos"
    )

    # Liberar memoria antes de la siguiente configuración
    del modelo_temporal
    del predicciones_validacion

    gc.collect()


CONFIGURACIÓN 1/3
regularizacion_fuerte
--------------------------------------------------
Entrenando modelo...
Generando predicciones...

RESULTADOS
----------
Accuracy:        0.8829
Precision macro: 0.7578
Recall macro:    0.6407
F1 macro:        0.6601
F1 negativo: 0.7967
F1 neutral: 0.2425
F1 positivo: 0.9411
Iteraciones realizadas: 17
¿Convergencia alcanzada? True
Tiempo de entrenamiento: 0.51 minutos
Tiempo de predicción: 0.07 segundos

CONFIGURACIÓN 2/3
regularizacion_intermedia
--------------------------------------------------
Entrenando modelo...
Generando predicciones...

RESULTADOS
----------
Accuracy:        0.8881
Precision macro: 0.7519
Recall macro:    0.6655
F1 macro:        0.6850
F1 negativo: 0.8077
F1 neutral: 0.3012
F1 positivo: 0.9460
Iteraciones realizadas: 19
¿Convergencia alcanzada? True
Tiempo de entrenamiento: 0.69 minutos
Tiempo de predicción: 0.12 segundos

CONFIGURACIÓN 3/3
clases_balanceadas
--------------------------------------------------
Entrenando 

c:\Users\Paula María\Desktop\IS\5º CURSO\TFG\Trabajo-Fin-de-Grado\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Generando predicciones...

RESULTADOS
----------
Accuracy:        0.8128
Precision macro: 0.6483
Recall macro:    0.7328
F1 macro:        0.6734
F1 negativo: 0.7523
F1 neutral: 0.3668
F1 positivo: 0.9011
Iteraciones realizadas: 500
¿Convergencia alcanzada? False
Tiempo de entrenamiento: 25.51 minutos
Tiempo de predicción: 0.09 segundos


In [10]:
resultados_ajuste_rl_df = pd.DataFrame(
    resultados_ajuste_rl
)

resultados_ajuste_rl_df = (
    resultados_ajuste_rl_df
    .sort_values(
        by="F1 macro",
        ascending=False
    )
    .reset_index(drop=True)
)

resultados_ajuste_rl_df.round(4)

,Configuración,C,class_weight,tol,max_iter,Iteraciones,Convergencia,Accuracy,Precision macro,Recall macro,F1 macro,F1 negativo,F1 neutral,F1 positivo,Tiempo entrenamiento,Tiempo predicción
0,regularizacion_intermedia,0.5,None,0.001,500,19,True,0.8881,0.7519,0.6655,0.6850,0.8077,0.3012,0.9460,41.2037,0.1224
1,clases_balanceadas,0.5,balanced,0.001,500,500,False,0.8128,0.6483,0.7328,0.6734,0.7523,0.3668,0.9011,1530.5585,0.0898
2,regularizacion_fuerte,0.1,None,0.001,500,17,True,0.8829,0.7578,0.6407,0.6601,0.7967,0.2425,0.9411,30.4772,0.0657


In [11]:
mejor_configuracion_rl = (
    resultados_ajuste_rl_df.iloc[0]
)

print("MEJOR CONFIGURACIÓN")
print("-------------------")

print(
    "Configuración:",
    mejor_configuracion_rl["Configuración"]
)

print(
    "F1 macro:",
    f"{mejor_configuracion_rl['F1 macro']:.4f}"
)

print(
    "Accuracy:",
    f"{mejor_configuracion_rl['Accuracy']:.4f}"
)

print(
    "F1 negativo:",
    f"{mejor_configuracion_rl['F1 negativo']:.4f}"
)

print(
    "F1 neutral:",
    f"{mejor_configuracion_rl['F1 neutral']:.4f}"
)

print(
    "F1 positivo:",
    f"{mejor_configuracion_rl['F1 positivo']:.4f}"
)

print(
    "Convergencia:",
    mejor_configuracion_rl["Convergencia"]
)

MEJOR CONFIGURACIÓN
-------------------
Configuración: regularizacion_intermedia
F1 macro: 0.6850
Accuracy: 0.8881
F1 negativo: 0.8077
F1 neutral: 0.3012
F1 positivo: 0.9460
Convergencia: True


In [12]:
# Entrenar modelo optimizado

from scipy.sparse import load_npz
import numpy as np

RUTA_VECTORIZACION = (
    RUTA_PROYECTO
    / "models"
    / "primera_iteracion"
    / "vectorizacion"
)

X_train_tfidf = load_npz(
    RUTA_VECTORIZACION / "X_train_tfidf.npz"
)

X_test_tfidf = load_npz(
    RUTA_VECTORIZACION / "X_test_tfidf.npz"
)

y_train = np.load(
    RUTA_VECTORIZACION / "y_train.npy",
    allow_pickle=True
)

y_test = np.load(
    RUTA_VECTORIZACION / "y_test.npy",
    allow_pickle=True
)



In [13]:
modelo_rl_optimizado = LogisticRegression(
    C=0.5,
    class_weight=None,
    solver="saga",
    tol=1e-3,
    max_iter=500,
    random_state=42
)

inicio_entrenamiento = perf_counter()

modelo_rl_optimizado.fit(
    X_train_tfidf,
    y_train
)

tiempo_entrenamiento_rl_optimizado = (
    perf_counter() - inicio_entrenamiento
)

print("ENTRENAMIENTO COMPLETADO")
print("------------------------")
print(
    "Tiempo:",
    f"{tiempo_entrenamiento_rl_optimizado:.2f} segundos"
)
print(
    "Iteraciones:",
    modelo_rl_optimizado.n_iter_
)
print(
    "¿Convergencia alcanzada?",
    modelo_rl_optimizado.n_iter_.max()
    < modelo_rl_optimizado.max_iter
)

ENTRENAMIENTO COMPLETADO
------------------------
Tiempo: 45.12 segundos
Iteraciones: [20]
¿Convergencia alcanzada? True


In [14]:
# Generar predicciones

inicio_prediccion = perf_counter()

predicciones_rl_optimizado = (
    modelo_rl_optimizado.predict(
        X_test_tfidf
    )
)

tiempo_prediccion_rl_optimizado = (
    perf_counter() - inicio_prediccion
)

print("PREDICCIÓN COMPLETADA")
print("---------------------")
print(
    "Reseñas clasificadas:",
    f"{len(predicciones_rl_optimizado):,}"
)
print(
    "Tiempo de predicción:",
    f"{tiempo_prediccion_rl_optimizado:.2f} segundos"
)
print(
    "¿Hay una predicción por reseña?",
    len(predicciones_rl_optimizado) == len(y_test)
)

PREDICCIÓN COMPLETADA
---------------------
Reseñas clasificadas: 297,069
Tiempo de predicción: 0.07 segundos
¿Hay una predicción por reseña? True


In [15]:
# Métricas generales

accuracy_rl_optimizado = accuracy_score(
    y_test,
    predicciones_rl_optimizado
)

precision_macro_rl_optimizado = precision_score(
    y_test,
    predicciones_rl_optimizado,
    average="macro",
    zero_division=0
)

recall_macro_rl_optimizado = recall_score(
    y_test,
    predicciones_rl_optimizado,
    average="macro",
    zero_division=0
)

f1_macro_rl_optimizado = f1_score(
    y_test,
    predicciones_rl_optimizado,
    average="macro",
    zero_division=0
)

print("REGRESIÓN LOGÍSTICA OPTIMIZADA")
print("------------------------------")
print(
    f"Accuracy:        "
    f"{accuracy_rl_optimizado:.4f}"
)
print(
    f"Precision macro: "
    f"{precision_macro_rl_optimizado:.4f}"
)
print(
    f"Recall macro:    "
    f"{recall_macro_rl_optimizado:.4f}"
)
print(
    f"F1 macro:        "
    f"{f1_macro_rl_optimizado:.4f}"
)

REGRESIÓN LOGÍSTICA OPTIMIZADA
------------------------------
Accuracy:        0.8891
Precision macro: 0.7547
Recall macro:    0.6706
F1 macro:        0.6901


In [16]:
# Métricas por clase

informe_rl_optimizado = classification_report(
    y_test,
    predicciones_rl_optimizado,
    labels=[
        "negativo",
        "neutral",
        "positivo"
    ],
    digits=4,
    output_dict=True,
    zero_division=0
)

informe_rl_optimizado_df = (
    pd.DataFrame(
        informe_rl_optimizado
    )
    .transpose()
)

informe_rl_optimizado_df.round(4)

,precision,recall,f1-score,support
negativo,0.8059,0.8180,0.8119,47697.0000
neutral,0.5380,0.2200,0.3123,23558.0000
positivo,0.9201,0.9739,0.9463,225814.0000
accuracy,0.8891,0.8891,0.8891,0.8891
macro avg,0.7547,0.6706,0.6901,297069.0000
weighted avg,0.8715,0.8891,0.8744,297069.0000


In [17]:
print(
    classification_report(
        y_test,
        predicciones_rl_optimizado,
        labels=[
            "negativo",
            "neutral",
            "positivo"
        ],
        digits=4,
        zero_division=0
    )
)

              precision    recall  f1-score   support

    negativo     0.8059    0.8180    0.8119     47697
     neutral     0.5380    0.2200    0.3123     23558
    positivo     0.9201    0.9739    0.9463    225814

    accuracy                         0.8891    297069
   macro avg     0.7547    0.6706    0.6901    297069
weighted avg     0.8715    0.8891    0.8744    297069



In [ ]:
# Comparación con la primera iteración
comparacion_rl = pd.DataFrame([
    {
        "Modelo": "Regresión Logística inicial",
        "Accuracy": 0.8889483588,
        "Precision macro": 0.7478088497,
        "Recall macro": 0.6755552591,
        "F1 macro": 0.6942251484,
        "Tiempo entrenamiento": 2291.2168,
        "Iteraciones": 1000,
        "Convergencia": False
    },
    {
        "Modelo": "Regresión Logística optimizada",
        "Accuracy": accuracy_rl_optimizado,
        "Precision macro": (
            precision_macro_rl_optimizado
        ),
        "Recall macro": (
            recall_macro_rl_optimizado
        ),
        "F1 macro": f1_macro_rl_optimizado,
        "Tiempo entrenamiento": (
            tiempo_entrenamiento_rl_optimizado
        ),
        "Iteraciones": int(
            modelo_rl_optimizado.n_iter_.max()
        ),
        "Convergencia": True
    }
])

comparacion_rl.round(4)

,Modelo,Accuracy,Precision macro,Recall macro,F1 macro,Tiempo entrenamiento,Iteraciones,Convergencia
0,Regresión Logística inicial,0.8889,0.7478,0.6756,0.6942,2291.2168,1000,False
1,Regresión Logística optimizada,0.8891,0.7547,0.6706,0.6901,45.1242,20,True


In [19]:
# Guardar RL optimizada
RUTA_RL_OPTIMIZADA = (
    RUTA_PROYECTO
    / "models"
    / "segunda_iteracion"
    / "regresion_logistica"
    / "modelo_optimizado"
)

RUTA_RL_OPTIMIZADA.mkdir(
    parents=True,
    exist_ok=True
)

In [23]:
# Modelos y predicciones

joblib.dump(
    modelo_rl_optimizado,
    RUTA_RL_OPTIMIZADA
    / "modelo_regresion_logistica_optimizado.joblib"
)

np.save(
    RUTA_RL_OPTIMIZADA
    / "predicciones_regresion_logistica_optimizado.npy",
    predicciones_rl_optimizado
)

In [26]:
# Métricas

metricas_rl_optimizado = {
    "modelo": "Regresión Logística optimizada",
    "C": 0.5,
    "class_weight": None,
    "solver": "saga",
    "tol": 1e-3,
    "max_iter": 500,
    "iteraciones_realizadas": 20,
    "convergencia_alcanzada": True,
    "accuracy": float(accuracy_rl_optimizado),
    "precision_macro": float(
        precision_macro_rl_optimizado
    ),
    "recall_macro": float(
        recall_macro_rl_optimizado
    ),
    "f1_macro": float(
        f1_macro_rl_optimizado
    ),
    "tiempo_entrenamiento_segundos": float(
        tiempo_entrenamiento_rl_optimizado
    ),
    "tiempo_prediccion_segundos": float(
        tiempo_prediccion_rl_optimizado
    ),
    "numero_resenas_entrenamiento": int(
        len(y_train)
    ),
    "numero_resenas_prueba": int(
        len(y_test)
    ),
    "numero_caracteristicas": int(
        X_train_tfidf.shape[1]
    )
}

with open(
    RUTA_RL_OPTIMIZADA
    / "metricas_regresion_logistica_optimizado.json",
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        metricas_rl_optimizado,
        archivo,
        indent=4,
        ensure_ascii=False
    )